In [7]:
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
import numpy as np
from custom.models.feedforward_neural_network import FeedForwardNeuralNetwork
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, mean_squared_error
from custom.util.data_manipulation import load_and_process_data
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform, loguniform

X, y, numeric_columns, categorical_columns = load_and_process_data("./data/claims_train.csv", False, None)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("mlp", FeedForwardNeuralNetwork(
            sizes_of_hidden_layers=[64], 
            epochs=1, 
            learning_rate=0.01
        )),
    ]
)

param_grid = {
    "mlp__sizes_of_hidden_layers": [
            [50,], [100,], [50, 50], [100, 50], [100, 100], [200, 100], [200, 200, 100, 50], [300, 200, 100, 50], [50, 50, 50], [10, 10, 10]
        ],
    "mlp__learning_rate": loguniform(1e-4, 1e-1),
    "mlp__epochs": [50, 100],
    "mlp__optimizer": ["adam"],
    "mlp__regularization_setting":(2,loguniform(1e-4, 1e-1))
}

grid = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_grid,
    n_iter=50, 
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose = 3
)

X_small = X.sample(n=50_000, random_state=42)
y_small = y.loc[X_small.index]

grid.fit(X_small, y_small)

print("Best parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


KeyboardInterrupt: 